# Security, project, and secure I/O tests

This notebook covers `boti.core.project`, `boti.core.security`, and the `SecureResource` behavior from `boti.core.secure_io`.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [2]:
from boti.core import ProjectService, SecureResource, is_secure_path
from boti.core.models import ResourceConfig

with tempfile.TemporaryDirectory() as tmp_dir:
    temp_root = Path(tmp_dir)
    (temp_root / "pyproject.toml").write_text("[project]\nname='demo'\n", encoding="utf-8")
    nested = temp_root / "src" / "nested"
    nested.mkdir(parents=True)

    detected = ProjectService.detect_project_root(start_path=nested)
    assert detected == temp_root.resolve()

    env_file = temp_root / ".env.local"
    env_file.write_text("BOTI_NOTEBOOK_TEST=loaded\n", encoding="utf-8")
    ProjectService.setup_environment(temp_root, env_file=env_file)
    assert os.environ["BOTI_NOTEBOOK_TEST"] == "loaded"

    inside = temp_root / "safe.txt"
    outside = Path("/etc/passwd")
    assert is_secure_path(inside, [temp_root])
    assert not is_secure_path(outside, [temp_root])

    resource = SecureResource(config=ResourceConfig(project_root=temp_root))
    resource.write_text_secure(inside, "hello notebook")
    assert resource.read_text_secure(inside) == "hello notebook"
    assert resource.get_secure_path(tempfile.gettempdir()) == Path(tempfile.gettempdir()).resolve()

    try:
        resource.get_secure_path(outside)
        raise AssertionError("expected PermissionError for outside path")
    except PermissionError:
        pass
    finally:
        resource.close()
        os.environ.pop("BOTI_NOTEBOOK_TEST", None)

print("Security and project checks passed.")

[2026-06-22 10:31:13][ERROR][SecureResource] SECURITY VIOLATION: Path traversal attempt detected. Target: /etc/passwd (resolved: /private/etc/passwd), Allowed Roots: [PosixPath('/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmpo3idew2w'), PosixPath('/private/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T')]
Security and project checks passed.
